In [30]:
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Import curated TTS2016 data

In [31]:
tts = pd.read_csv('../Travel_data/TTS_dataset/tts_data_aggregated.csv')

# Identify mid-trip timestamps and extract hour integers
tts.timestamp = pd.to_datetime(tts.timestamp)
def midtime(a, b):
    return a + timedelta(minutes=b/2)

tts['mid_time'] = tts.apply(lambda row: midtime(row['timestamp'],row['trip_time']), axis=1)

def endtime(a, b):
    return a + timedelta(minutes=b)

tts['end_time'] = tts.apply(lambda row: endtime(row['timestamp'],row['trip_time']), axis=1)

def to_integer(dt_time):
    return dt_time.hour

tts['int_time'] = tts.mid_time.apply(to_integer)

tts['int_endtime'] = tts.end_time.apply(to_integer)


# Trip distributions - TTS2016

TTS only surveyed weekday travel habits, excluding weekends from their data.

## Weekday travel schedule (*trip_by_time.csv*)

In [32]:
tts.int_time.value_counts(normalize=True).sort_index() * 100

int_time
0      0.378448
1      0.150453
2      0.113129
3      0.067704
4      0.369479
5      1.304604
6      3.349034
7      8.117249
8     10.185984
9      5.600627
10     4.860223
11     4.963226
12     4.597508
13     4.485826
14     5.550862
15     7.415327
16     8.973098
17     9.758059
18     6.898577
19     4.738703
20     3.249214
21     2.398574
22     1.521605
23     0.952486
Name: proportion, dtype: float64

In [33]:
fig = px.histogram(tts, x='int_endtime', 
                #    facet_col='WEEKDAY', 
                   nbins=24, text_auto='.1f',
                   histnorm='percent'
                   )

fig.update_xaxes(title='Start time (hour)',
                 dtick=1
                 )

fig.update_yaxes(title='Relative freq. (%)')

fig.update_layout(width=900,
                  height=500,
                  template='plotly_white',
                  bargap=0.05,
                #   barmode='stack',
                  title={'text':"TTS 2016 Trip start time distribution during weekdays",
                         'x':0.5,
                         'y':0.97},
                #   legend=dict(orientation='v',
                #               y=1,
                #               x=0.85),
                font=dict(
                       # family="Courier New, monospace",
                       size=13,
                       # color="#ffffff"
                       )
                )

fig.show()

# Avg. trip characteristics - TTS2016

## Avg. total daily driven distance in weekdays (*d_tot.csv*)

In [34]:
# Ran tts_data.groupby('hp_num')['trip_man_km'].sum().mean() on the VM as identifiers are not allowed to be exported.
print('38.65 km')

38.65 km


# Occupation groups - TTS2016

In [35]:
tts_occupation = tts[(tts.purp_orig == 'S') | (tts.purp_orig == 'W') |
                     (tts.purp_dest == 'S') | (tts.purp_dest == 'W') | (tts.purp_dest == 'C') | (tts.purp_dest == 'R')]

tts_personal = tts[(tts.purp_orig != 'S') & (tts.purp_orig != 'W') &
                     (tts.purp_dest != 'S') & (tts.purp_dest != 'W') & (tts.purp_dest != 'C') & (tts.purp_dest != 'R')]

### *d_min.csv* - occupation, weekday

In [36]:
print("{} km".format(round(tts_occupation.trip_man_km.mean(), 8)))

18.9536318 km


### *d_min.csv* - personal, weekday

In [37]:
print("{} km".format(round(tts_personal.trip_man_km.mean(), 8)))

8.83333058 km


### *t_func.csv* - occupation, weekday

In [38]:
print("{} min".format(round(tts_occupation.trip_time.mean(), 8)))

23.0595335 min


### *t_func.csv* - personal, weekday

In [39]:
print("{} min".format(round(tts_personal.trip_time.mean(), 8)))

9.96679199 min


# Time windows - TTS2016

## Worker occupation

In [40]:
tts_worker = tts[(tts.purp_orig == 'W') | (tts.purp_dest == 'W') | (tts.purp_dest == 'R')]

In [41]:
fig = px.histogram(tts_worker, x='timestamp', 
                   nbins=48, 
                   text_auto='.1f',
                   histnorm='percent'
                   )

fig.add_vrect(x0=datetime(2016, 1, 1, 6, 0),   # 6:00 - 9:30
              x1=datetime(2016, 1, 1, 9, 30),
              annotation_text='Main window: +3%',
              annotation_position='top left',
              fillcolor="green", opacity=0.2, line_width=0)

fig.add_vrect(x0=datetime(2016, 1, 1, 15, 0),   # 15:00 - 18:30
              x1=datetime(2016, 1, 1, 18, 30),
              annotation_text='Main window: +3%',
              annotation_position='top left',
              fillcolor="green", opacity=0.2, line_width=0)

fig.update_xaxes(title='Start time',
                 range=[datetime(2016, 1, 1, 0, 0), datetime(2016, 1, 1, 23, 0)],
                 showgrid=True,
                 tickformat="%H:%M",
                 dtick=3600*1000    # dtick is in miliseconds
                 )

fig.update_layout(width=1200,
                  height=500,
                  template='simple_white',
                  bargap=0.05,
                #   barmode='stack',
                  title={'text':"TTS 2016 Trips to/from work - functioning windows",
                         'x':0.5,
                         'y':0.95},
                #   legend=dict(orientation='v',
                #               y=1,
                #               x=0.85)
                )

fig.show()

## Student occupation

In [42]:
tts_student = tts[(tts.purp_orig == 'S') | (tts.purp_dest == 'S') | (tts.purp_dest == 'C')]

In [43]:
fig = px.histogram(tts_student, x='timestamp', 
                   nbins=48, 
                   text_auto='.1f',
                   histnorm='percent'
                   )

fig.add_vrect(x0=datetime(2016, 1, 1, 7, 0),   # 7:00 - 9:30
              x1=datetime(2016, 1, 1, 9, 30),
              annotation_text='Main window: +3%',
              annotation_position='top left',
              fillcolor="green", opacity=0.2, line_width=0)

fig.add_vrect(x0=datetime(2016, 1, 1, 14, 0),   # 14:00 - 18:30
              x1=datetime(2016, 1, 1, 18, 30),
              annotation_text='Main window: +3%',
              annotation_position='top left',
              fillcolor="green", opacity=0.2, line_width=0)

fig.update_xaxes(title='Start time',
                 range=[datetime(2016, 1, 1, 0, 0), datetime(2016, 1, 1, 23, 0)],
                 showgrid=True,
                 tickformat="%H:%M",
                 dtick=3600*1000    # dtick is in miliseconds
                 )

fig.update_layout(width=1200,
                  height=500,
                  template='simple_white',
                  bargap=0.05,
                #   barmode='stack',
                  title={'text':"TTS 2016 Trips to/from school - functioning windows",
                         'x':0.5,
                         'y':0.95},
                #   legend=dict(orientation='v',
                #               y=1,
                #               x=0.85)
                )

fig.show()

## No occupation - not worker nor student

In [44]:
fig = px.histogram(tts_personal, x='timestamp', 
                   nbins=48, 
                   text_auto='.1f',
                   histnorm='percent'
                   )

fig.add_vrect(x0=datetime(2016, 1, 1, 7, 30),   # 7:30 - 20:30
              x1=datetime(2016, 1, 1, 20, 30),
              annotation_text='Main window: +2%',
              annotation_position='top left',
              fillcolor="green", opacity=0.2, line_width=0)

fig.update_xaxes(title='Start time',
                 range=[datetime(2016, 1, 1, 0, 0), datetime(2016, 1, 1, 23, 0)],
                 showgrid=True,
                 tickformat="%H:%M",
                 dtick=3600*1000    # dtick is in miliseconds
                 )

fig.update_layout(width=1200,
                  height=500,
                  template='simple_white',
                  bargap=0.05,
                #   barmode='stack',
                  title={'text':"TTS 2016 Trips to/from any activity except work or school - functioning windows",
                         'x':0.5,
                         'y':0.95},
                #   legend=dict(orientation='v',
                #               y=1,
                #               x=0.85)
                )

fig.show()

# NHTS 2022 - matching weekend travel patterns

In [45]:
data_NHTS = pd.read_csv('../Travel_data/NHTS2022/trippub.csv')

data_NHTS = data_NHTS[['HOUSEID','PERSONID','TRIPID','FRSTHM','TRAVDAY','LOOP_TRIP','DWELTIME','WHYFROM',
                       'WHYTRP1S','TRVLCMIN','STRTTIME','ENDTIME','TRPHHVEH','VEHID','TRPTRANS','WHYTO','TRPMILES',
                       'VMT_MILE','CDIVMSAR','HHFAMINC','HHSIZE','HHVEHCNT','MSASIZE', 'MSACAT', 'URBRUR','WRKCOUNT','WORKER','DRIVER','PROXY','VEHTYPE']]

data_NHTS.drop_duplicates(inplace=True)

data_NHTS = data_NHTS[data_NHTS.TRPTRANS <= 4]  # Exclude modes other than cars, van, SUV/crossover and pickup truck
data_NHTS = data_NHTS[data_NHTS.VEHTYPE > 0]  # Exclude vehicle types other than cars, van, SUV/crossover and pickup truck
data_NHTS = data_NHTS[data_NHTS.VEHTYPE <= 4]  # Exclude vehicle types other than cars, van, SUV/crossover and pickup truck
data_NHTS = data_NHTS[data_NHTS.DRIVER == 1]    # Only trips carried out as a driver, not as a passenger
data_NHTS = data_NHTS[data_NHTS.TRPHHVEH == 1]  # Exclude all non-household vehicles
data_NHTS = data_NHTS[data_NHTS.VEHID != -1]    # Exclude all respondents that skipped this question
data_NHTS = data_NHTS[data_NHTS.TRPMILES > 0]   # Exclude trips with no clear response
data_NHTS = data_NHTS[data_NHTS.TRPMILES < 1280] # Exclude all trips with distance travelled of more than 1280 miles 
data_NHTS = data_NHTS[data_NHTS.VMT_MILE > 0]   # Exclude trips with no clear response
data_NHTS = data_NHTS[data_NHTS.VMT_MILE < 1280] # Exclude all trips with distance travelled of more than 1280 miles 
data_NHTS = data_NHTS[data_NHTS.TRVLCMIN > 0]   # Exclude trips with no clear response
data_NHTS = data_NHTS[data_NHTS.TRVLCMIN < 960] # Exclude trips with durations of more than 16 hours
data_NHTS = data_NHTS[data_NHTS.TRVLCMIN > 0]   # Exclude trips with no clear response

data_NHTS = data_NHTS[data_NHTS.WHYFROM >= 1] # Exclude trips with no clear response
data_NHTS = data_NHTS[data_NHTS.WHYFROM < 97] # Exclude trips with no clear response
data_NHTS = data_NHTS[data_NHTS.WHYTO >= 1] # Exclude trips with no clear response
data_NHTS = data_NHTS[data_NHTS.WHYTO < 97] # Exclude trips with no clear response
data_NHTS = data_NHTS[data_NHTS.STRTTIME >= 0] # Exclude trips with no clear response

data_NHTS['AVGSPEED'] = data_NHTS.TRPMILES / (data_NHTS.TRVLCMIN/60)
data_NHTS = data_NHTS[data_NHTS.AVGSPEED >= 0.5] # Exclude trips calculated avg. speed lower than 0.5 mph
data_NHTS = data_NHTS[data_NHTS.AVGSPEED < 120] # Exclude trips calculated avg. speed higher than 120 mph

veh_NHTS = pd.read_csv('../Travel_data/NHTS2022/vehpub.csv')
veh_NHTS = veh_NHTS[['HOUSEID', 'VEHID', 'VEHFUEL']]
curated_NHTS = pd.merge(data_NHTS, veh_NHTS, on=['HOUSEID', 'VEHID'], how='left')

In [46]:
curated_NHTS['START_TIME'] = pd.to_datetime('2022-01-01 ' + curated_NHTS.STRTTIME.astype(str).str.zfill(4))

def midtime(a, b):
    return a + timedelta(minutes=b/2)

curated_NHTS['MIDTIME'] = curated_NHTS.apply(lambda row: midtime(row['START_TIME'],row['TRVLCMIN']), axis=1)

def to_integer(dt_time):
    return dt_time.hour

curated_NHTS['INTTIME'] = curated_NHTS.MIDTIME.apply(to_integer)

days = {
    1:'sunday',
    2:'weekday',
    3:'weekday',
    4:'weekday',
    5:'weekday',
    6:'weekday',
    7:'saturday'
}

curated_NHTS['WEEKDAY'] = curated_NHTS.TRAVDAY.map(days)

purpose = {
    1:'personal',
    10:'occupation',
    20:'occupation',
    30:'personal',
    40:'personal',
    50:'personal',
    70:'personal',
    80:'personal',
    97:'personal',
}

curated_NHTS['PURPOSE'] = curated_NHTS.WHYTRP1S.map(purpose)

In [47]:
fig = go.Figure()

fig.add_trace(go.Histogram(x=tts.int_time,
                           nbinsx=24,
                           histnorm='percent',
                           texttemplate='%{y:.1f}',
                           name= 'TTS 2016'
                           ))

fig.add_trace(go.Histogram(x=curated_NHTS[(curated_NHTS.WEEKDAY == 'weekday')].INTTIME,
                           nbinsx=24,
                           histnorm='percent',
                           texttemplate='%{y:.1f}',
                           name= 'NHTS 2022'
                           ))

fig.add_trace(go.Histogram(x=curated_NHTS[(curated_NHTS.WEEKDAY == 'weekday') & (curated_NHTS.CDIVMSAR == 21)].INTTIME,
                           nbinsx=24,
                           histnorm='percent',
                           texttemplate='%{y:.1f}',
                           name= 'NHTS 2022 - Mid-Atlantic'
                           ))

fig.update_xaxes(title='Start time (hour)',
                 dtick=1
                 )

fig.update_layout(width=1400,
                  height=500,
                  template='plotly_white',
                  bargap=0.1,
                  bargroupgap=0.05,
                #   barmode='stack',
                  title={'text':"Trip start time distributions in weekdays",
                         'x':0.5,
                         'y':0.97},
                  legend=dict(orientation='v',
                              y=1,
                              x=0.85)
                )

fig.show()

In [48]:
# Ran tts_data.groupby('hp_num')['trip_man_km'].sum().mean() on the VM as identifiers are not allowed to be exported.
print('TTS avg. daily total distance - weekdays: 38.65 km')
print('NHTS avg. daily total distance - weekdays: {} km'.format(
      round(curated_NHTS[(curated_NHTS.WEEKDAY == 'weekday') & (curated_NHTS.CDIVMSAR == 21)].groupby(['HOUSEID', 'PERSONID'], as_index=False)['TRPMILES'].sum().TRPMILES.mean()*1.60934, 2)))
print('NHTS avg. daily total distance - saturday: {} km'.format(
      round(curated_NHTS[(curated_NHTS.WEEKDAY == 'saturday') & (curated_NHTS.CDIVMSAR == 21)].groupby(['HOUSEID', 'PERSONID'], as_index=False)['TRPMILES'].sum().TRPMILES.mean()*1.60934, 2)))
print('NHTS avg. daily total distance - sunday: {} km'.format(
      round(curated_NHTS[(curated_NHTS.WEEKDAY == 'sunday') & (curated_NHTS.CDIVMSAR == 21)].groupby(['HOUSEID', 'PERSONID'], as_index=False)['TRPMILES'].sum().TRPMILES.mean()*1.60934, 2)))

TTS avg. daily total distance - weekdays: 38.65 km
NHTS avg. daily total distance - weekdays: 45.4 km
NHTS avg. daily total distance - saturday: 39.58 km
NHTS avg. daily total distance - sunday: 42.69 km


In [49]:
print("TTS occupation - weekdays: {} km".format(round(tts_occupation.trip_man_km.mean(), 8)))
print("TTS personal - weekdays: {} km".format(round(tts_personal.trip_man_km.mean(), 8)))
print("TTS occupation - weekdays: {} min".format(round(tts_occupation.trip_time.mean(), 8)))
print("TTS personal - weekdays: {} min".format(round(tts_personal.trip_time.mean(), 8)))

TTS occupation - weekdays: 18.9536318 km
TTS personal - weekdays: 8.83333058 km
TTS occupation - weekdays: 23.0595335 min
TTS personal - weekdays: 9.96679199 min


In [50]:
print("NHTS occupation - weekdays: {} km".format(round(curated_NHTS[(curated_NHTS.PURPOSE == 'occupation') & (curated_NHTS.WEEKDAY == 'weekday') & (curated_NHTS.CDIVMSAR == 21)].TRPMILES.mean()*1.60934, 8)))
print("NHTS personal - weekdays: {} km".format(round(curated_NHTS[(curated_NHTS.PURPOSE == 'personal') & (curated_NHTS.WEEKDAY == 'weekday') & (curated_NHTS.CDIVMSAR == 21)].TRPMILES.mean()*1.60934, 8)))
print("NHTS occupation - weekdays: {} min".format(round(curated_NHTS[(curated_NHTS.PURPOSE == 'occupation') & (curated_NHTS.WEEKDAY == 'weekday') & (curated_NHTS.CDIVMSAR == 21)].TRVLCMIN.mean(), 8)))
print("NHTS personal - weekdays: {} min".format(round(curated_NHTS[(curated_NHTS.PURPOSE == 'personal') & (curated_NHTS.WEEKDAY == 'weekday') & (curated_NHTS.CDIVMSAR == 21)].TRVLCMIN.mean(), 8)))

NHTS occupation - weekdays: 20.6076242 km
NHTS personal - weekdays: 15.61976871 km
NHTS occupation - weekdays: 28.25409836 min
NHTS personal - weekdays: 22.4247619 min


## Household income matching

In [51]:
income_nhts = curated_NHTS[(curated_NHTS.HHFAMINC != -7) & (curated_NHTS.HHFAMINC != -8)]

#  Using USD to CAD 2020 exchange rate from: https://www.bankofcanada.ca/rates/exchange/annual-average-exchange-rates/
income = {
    1:'Less than $13,000',
    2:'$13,400 to $20,000',
    3:'$20,000 to $34,000',
    4:'$34,500 to $47,000',
    5:'$47,000 to $67,000',
    6:'$67,000 to $101,000',
    7:'$101,000 to $134,000',
    8:'$134,000 to $168,000',
    9:'$168,000 to $201,000',
    10:'$201,000 or more',
    11:'$201,000 or more'
}

income_nhts['HINCOME'] = income_nhts.HHFAMINC.map(income)

C:\Users\rashi\AppData\Local\Temp\ipykernel_19480\1677985214.py:18: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [52]:
income_order = ['Less than $13,000', '$13,400 to $20,000', '$20,000 to $34,000', '$34,500 to $47,000', '$47,000 to $67,000', 
                '$67,000 to $101,000', '$101,000 to $134,000', '$134,000 to $168,000', '$168,000 to $201,000', '$201,000 or more']

income_nhts['HINCOME'] = pd.Categorical(income_nhts['HINCOME'], categories=income_order, ordered=True)

C:\Users\rashi\AppData\Local\Temp\ipykernel_19480\1635875588.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [53]:
income_tts = pd.read_csv('../Travel_data/2021census_household-income_TTS-CMAs.csv', usecols=['GEO', 'Household total income groups (22)', 'VALUE'])
income_tts.columns = ['geo', 'hincome', 'counts']

income_tts = income_tts.groupby('hincome').counts.sum().reset_index()

income_aggrupation = {
    'Under $5,000':'Less than $13,000', 
    '$5,000 to $9,999':'Less than $13,000', 
    '$10,000 to $14,999':'$13,400 to $20,000',
       '$15,000 to $19,999':'$13,400 to $20,000', 
       '$20,000 to $24,999':'$20,000 to $34,000', 
       '$25,000 to $29,999':'$20,000 to $34,000',
       '$30,000 to $34,999':'$20,000 to $34,000', 
       '$35,000 to $39,999':'$34,500 to $47,000', 
       '$40,000 to $44,999':'$34,500 to $47,000',
       '$45,000 to $49,999':'$47,000 to $67,000', 
       '$50,000 to $59,999':'$47,000 to $67,000', 
       '$60,000 to $69,999':'$47,000 to $67,000',
       '$70,000 to $79,999':'$67,000 to $101,000', 
       '$80,000 to $89,999':'$67,000 to $101,000', 
       '$90,000 to $99,999':'$67,000 to $101,000',
       '$100,000 to $124,999':'$101,000 to $134,000', 
       '$125,000 to $149,999':'$134,000 to $168,000',
       '$150,000 to $199,999':'$168,000 to $201,000', 
       '$200,000 and over':'$201,000 or more'
}

income_tts['nhts_income'] = income_tts.hincome.map(income_aggrupation)

income_tts['nhts_income'] = pd.Categorical(income_tts['nhts_income'], categories=income_order, ordered=True)
income_tts = income_tts.groupby('nhts_income').counts.sum().reset_index()

C:\Users\rashi\AppData\Local\Temp\ipykernel_19480\228717636.py:31: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [54]:
fig = px.histogram(income_tts, x='nhts_income', y='counts', 
                   histnorm='percent',
                   text_auto='.1f')

fig.add_trace(go.Histogram(x=income_nhts[income_nhts.CDIVMSAR == 11].HINCOME, 
                           histnorm='percent',
                           texttemplate='%{y:.1f}',
                           name='NHTS 2022 - Mid-Atlantic'))

fig.update_xaxes(title='Household income groups (CAD)',
                 )

fig.update_layout(width=1200,
                  height=500,
                  template='plotly_white',
                  bargap=0.1,
                  bargroupgap=0.05,
                  barmode='group',
                  title={'text':"Trip start time distributions in weekdays",
                         'x':0.5,
                         'y':0.97},
                  legend=dict(orientation='v',
                              y=1.05,
                              x=0.85)
                )

fig.show()

## Weekend data from NHTS 2022

In [55]:
curated_NHTS[(curated_NHTS.WEEKDAY == 'saturday') & (curated_NHTS.CDIVMSAR == 21)].INTTIME.value_counts(normalize=True).sort_index() * 100

INTTIME
0      0.970874
1      0.970874
6      0.970874
7      1.941748
8      2.912621
9      3.883495
10    12.621359
11     9.708738
12     2.912621
13     7.766990
14     6.796117
15    10.679612
16     4.854369
17     7.766990
18     6.796117
19     2.912621
20     6.796117
21     1.941748
22     2.912621
23     3.883495
Name: proportion, dtype: float64

In [56]:
curated_NHTS[(curated_NHTS.WEEKDAY == 'sunday') & (curated_NHTS.CDIVMSAR == 21)].INTTIME.value_counts(normalize=True).sort_index() * 100

INTTIME
6      0.819672
7      0.819672
8      4.918033
9     13.114754
10     6.557377
11    10.655738
12    13.114754
13     9.836066
14     9.836066
15     4.098361
16     7.377049
17     3.278689
18     4.918033
19     4.098361
20     3.278689
21     2.459016
22     0.819672
Name: proportion, dtype: float64

# Distance groups - TTS 2016

In [57]:
p33 = np.percentile(tts.trip_man_km, 33)
p66 = np.percentile(tts.trip_man_km, 66)

In [58]:
fig = go.Figure(go.Histogram(x=tts.trip_man_km, cumulative_enabled=True, 
                             histnorm='probability'))

fig.update_xaxes(title='Trip manhattan distance (km)',
                 dtick=2,
                 range=[0, 40]
                 )

fig.add_vline(x=p33,
              line_dash='dash',
              line_width=2,
              line_color='black',
              annotation_position='top left',
              annotation_text='33th Percentile')

fig.add_vline(x=p66,
              line_dash='dash',
              line_width=2,
              line_color='black',
              annotation_position='top left',
              annotation_text='66th Percentile')

fig.update_layout(width=1000,
                  height=500,
                  template='simple_white',
                #   bargap=0.05,
                #   bargroupgap=0.05,
                #   barmode='stack',
                  title={'text':"TTS 2016 trip distance percentiles",
                         'x':0.5,
                         'y':0.9},
                  # legend=dict(orientation='v',
                  #             y=1,
                  #             x=0.85),
                  font=dict(
                       # family="Courier New, monospace",
                       size=13,
                       # color="#ffffff"
                       )
                )

fig.show()